# Lab 04: Eigendecomposition

**Reference:** Goodfellow, Bengio & Courville, *Deep Learning* -- Chapter 2, Section 2.7

---

Eigendecomposition is arguably the single most important matrix factorization in machine learning. It reveals the **intrinsic geometry** of a linear transformation: the directions along which the transformation acts by pure scaling, and the scale factors themselves. Understanding eigendecomposition is the gateway to PCA, spectral methods, optimization landscape analysis, and much more.

**Why should you care?**
- PCA (the most common dimensionality reduction) is literally the eigendecomposition of the covariance matrix.
- Whether a critical point in your loss landscape is a minimum, maximum, or saddle point depends entirely on the Hessian's eigenvalues.
- The condition number $\kappa = \lambda_{\max}/\lambda_{\min}$ determines how hard a problem is to optimize with gradient descent.
- Spectral clustering, Google's PageRank, and quantum mechanics all rest on eigendecomposition.

**Learning objectives:**
1. Understand the geometric meaning of eigenvectors and eigenvalues
2. Derive eigenvalues from the characteristic polynomial
3. Implement and verify the decomposition $A = V\,\text{diag}(\boldsymbol{\lambda})\,V^{-1}$
4. Apply the Spectral Theorem: real symmetric matrices have real eigenvalues and orthogonal eigenvectors
5. Classify matrices (PD, PSD, indefinite) via eigenvalue signs
6. Visualize quadratic forms and their principal axes
7. Implement power iteration and the QR algorithm from scratch
8. Connect everything to PCA, Hessians, spectral clustering, and condition numbers

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch
import matplotlib.gridspec as gridspec
from numpy import linalg as LA

np.random.seed(42)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print('NumPy version:', np.__version__)
print('Setup complete.')

NumPy version: 1.26.4
Setup complete.


---

## Part 1: Eigenvectors and Eigenvalues -- Definition and Geometric Intuition

### The Core Definition

An **eigenvector** of a square matrix $A$ is a **nonzero** vector $\mathbf{v}$ such that multiplication by $A$ only **scales** it:

$$A\mathbf{v} = \lambda \mathbf{v}$$

The scalar $\lambda$ is the corresponding **eigenvalue**. The word "eigen" is German for "own" or "characteristic" -- eigenvectors are the matrix's *own* special directions.

### Geometric Intuition

Think about what a matrix $A$ does to an arbitrary vector $\mathbf{x}$. Generically, $A\mathbf{x}$ points in a **different direction** than $\mathbf{x}$ -- the matrix both rotates and stretches it. But there are special directions where the matrix only stretches (or compresses, or flips). Those are the eigenvector directions.

- If $\lambda > 1$: the eigenvector is **stretched** (magnified)
- If $0 < \lambda < 1$: the eigenvector is **compressed** (shrunk)
- If $\lambda = 1$: the eigenvector is **unchanged**
- If $\lambda < 0$: the eigenvector is **flipped** (reversed direction) and scaled by $|\lambda|$
- If $\lambda = 0$: the eigenvector is **sent to zero** (the matrix is singular along this direction)

**Key insight from Goodfellow:** If we know the eigenvectors and eigenvalues of $A$, we can understand the matrix's behavior completely. Instead of thinking of $A$ as a complicated transformation, we can decompose it into simple scalings along independent directions.

**Note:** Any nonzero scalar multiple of an eigenvector is also an eigenvector (same eigenvalue). So when we refer to "the" eigenvector, we usually mean a unit-norm version.

### Exercise 1 -- Verify Eigendecomposition: $A = V\,\text{diag}(\boldsymbol{\lambda})\,V^{-1}$

**The Eigendecomposition** (Goodfellow Eq. 2.40): If $A$ has $n$ linearly independent eigenvectors, we concatenate them as columns of a matrix $V = [\mathbf{v}_1 | \mathbf{v}_2 | \cdots | \mathbf{v}_n]$ and collect eigenvalues into a diagonal matrix $\Lambda = \text{diag}(\boldsymbol{\lambda})$. Then:

$$A = V \Lambda V^{-1}$$

**Why does this work?** Start from the $n$ eigenvector equations $A\mathbf{v}_i = \lambda_i \mathbf{v}_i$. Stack them:

$$A [\mathbf{v}_1 | \cdots | \mathbf{v}_n] = [\lambda_1 \mathbf{v}_1 | \cdots | \lambda_n \mathbf{v}_n]$$

The right-hand side is exactly $V\Lambda$, so $AV = V\Lambda$. If $V$ is invertible (which requires $n$ linearly independent eigenvectors), multiply both sides on the right by $V^{-1}$:

$$A = V \Lambda V^{-1}$$

**Not every matrix can be eigendecomposed.** You need $n$ linearly independent eigenvectors. Defective matrices (e.g., $\begin{pmatrix} 0 & 1 \\ 0 & 0 \end{pmatrix}$) don't have enough independent eigenvectors.

**Task:** Compute the eigendecomposition of the matrix below, reconstruct $A$ from $V\Lambda V^{-1}$, and verify that each eigenvector equation $A\mathbf{v}_i = \lambda_i \mathbf{v}_i$ holds.

In [2]:
# Exercise 1: Verify eigendecomposition A = V diag(lambda) V^{-1}

A = np.array([[4.0, 1.0],
              [2.0, 3.0]])

# Step 1: Compute eigenvalues and eigenvectors
lambdas, V = np.linalg.eig(A)   # lambdas shape (2,), V shape (2,2), columns are eigenvectors

# Step 2: Build diagonal matrix Lambda
Lambda = np.diag(lambdas)        # shape (2,2)

# Step 3: Reconstruct A from V, Lambda, V_inv
A_rec = V @ Lambda @ np.linalg.inv(V)

# Step 4: Verify each eigenvector equation A v_i = lambda_i v_i
eigvec_checks = []
for i in range(len(lambdas)):
    Av_i = A @ V[:, i]
    lv_i = lambdas[i] * V[:, i]
    eigvec_checks.append((Av_i, lv_i))

print('Matrix A:')
print(A)
print('\nEigenvalues:', lambdas)
print('\nEigenvectors (columns of V):')
print(V)
print('\nLambda (diagonal):')
print(Lambda)
print('\nReconstructed A:')
print(A_rec)
print('\nReconstruction error:', np.max(np.abs(A - A_rec)))
for i, (Av_i, lv_i) in enumerate(eigvec_checks):
    print(f'  A*v_{i+1} = {np.round(Av_i,6)},  lambda_{i+1}*v_{i+1} = {np.round(lv_i,6)},  match: {np.allclose(Av_i, lv_i)}')

Matrix A:
[[4. 1.]
 [2. 3.]]

Eigenvalues: [5. 2.]

Eigenvectors (columns of V):
[[ 0.70710678 -0.4472136 ]
 [ 0.70710678  0.89442719]]

Lambda (diagonal):
[[5. 0.]
 [0. 2.]]

Reconstructed A:
[[4. 1.]
 [2. 3.]]

Reconstruction error: 8.881784197001252e-16
  A*v_1 = [3.535534 3.535534],  lambda_1*v_1 = [3.535534 3.535534],  match: True
  A*v_2 = [-0.894427  1.788854],  lambda_2*v_2 = [-0.894427  1.788854],  match: True


In [3]:
# Verification cell -- do not modify
assert lambdas is not None, 'Compute eigenvalues'
assert V is not None, 'Compute eigenvectors'
assert Lambda is not None, 'Build Lambda'
assert A_rec is not None, 'Compute A_rec'
assert Lambda.shape == (2, 2), f'Lambda shape should be (2,2), got {Lambda.shape}'
assert np.allclose(np.diag(Lambda), np.sort(np.abs(lambdas)), atol=1e-8) or \
       set(np.round(np.diag(Lambda), 6)) == set(np.round(lambdas, 6)), \
       'Lambda diagonal should contain eigenvalues'
assert np.allclose(A, A_rec, atol=1e-8), f'Reconstruction failed. Max error: {np.max(np.abs(A - A_rec)):.2e}'
for i in range(len(lambdas)):
    Av_i = A @ V[:, i]
    lv_i = lambdas[i] * V[:, i]
    assert np.allclose(Av_i, lv_i, atol=1e-8), f'Eigenvector equation failed for i={i}'
print('Exercise 1 PASSED: A = V diag(lambda) V^-1 verified, all eigenvector equations hold.')

Exercise 1 PASSED: A = V diag(lambda) V^-1 verified, all eigenvector equations hold.


### Visualizing the Geometry: Eigenvectors Are the Invariant Directions

Let's see this in action. We take a circle of unit vectors, apply $A$, and watch how the circle deforms into an ellipse. The eigenvectors are the directions that stay on their original line (they only get scaled).

In [4]:
# Visualize: how A transforms the unit circle
# The eigenvectors are the directions that remain unchanged (only scaled)

theta = np.linspace(0, 2 * np.pi, 300)
circle = np.vstack([np.cos(theta), np.sin(theta)])  # 2 x 300

# Apply A to every point on the unit circle
transformed = A @ circle  # 2 x 300

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: Before transformation
ax = axes[0]
ax.plot(circle[0], circle[1], 'b-', lw=2, label='Unit circle')
for i in range(len(lambdas)):
    v = V[:, i]
    ax.annotate('', xy=v, xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color=f'C{i+1}', lw=2.5))
    ax.text(v[0]*1.15, v[1]*1.15, f'$v_{i+1}$', fontsize=14, color=f'C{i+1}', fontweight='bold')
ax.set_xlim(-6, 6); ax.set_ylim(-6, 6)
ax.set_aspect('equal')
ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)
ax.set_title('Before: Unit circle + eigenvectors', fontsize=13)
ax.legend(fontsize=11)

# Right: After transformation
ax = axes[1]
ax.plot(transformed[0], transformed[1], 'b-', lw=2, label='$A \\times$ circle')
for i in range(len(lambdas)):
    v_scaled = lambdas[i] * V[:, i]  # A*v = lambda*v
    ax.annotate('', xy=v_scaled, xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color=f'C{i+1}', lw=2.5))
    ax.text(v_scaled[0]*1.1, v_scaled[1]*1.1,
            f'$\\lambda_{i+1} v_{i+1}$  ($\\lambda={lambdas[i]:.0f}$)',
            fontsize=12, color=f'C{i+1}', fontweight='bold')
ax.set_xlim(-6, 6); ax.set_ylim(-6, 6)
ax.set_aspect('equal')
ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)
ax.set_title('After: Transformed ellipse + scaled eigenvectors', fontsize=13)
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()

<Figure size 1400x600 with 2 Axes>

**What the plot shows:** The unit circle (left) gets deformed into an ellipse (right) by $A$. Notice that the eigenvectors stay on the same line through the origin -- they just get longer or shorter. The eigenvalue tells you the stretch factor. Every other direction gets rotated *and* stretched.

This is the fundamental geometric insight: **eigenvectors are the axes of the ellipse that the matrix carves out from the unit circle.**

---

## Part 2: The Characteristic Polynomial

### Where Do Eigenvalues Come From?

We need to find $\lambda$ such that $A\mathbf{v} = \lambda \mathbf{v}$ for some nonzero $\mathbf{v}$. Rearranging:

$$(A - \lambda I)\mathbf{v} = \mathbf{0}$$

This has a nonzero solution $\mathbf{v}$ if and only if $(A - \lambda I)$ is **singular**, which means:

$$\det(A - \lambda I) = 0$$

This is the **characteristic equation**. For an $n \times n$ matrix, it is a degree-$n$ polynomial in $\lambda$, so there are exactly $n$ roots (counting multiplicity, possibly complex).

### The 2x2 Case (Closed Form)

For $A = \begin{pmatrix} a & b \\ c & d \end{pmatrix}$:

$$\det\begin{pmatrix} a - \lambda & b \\ c & d - \lambda \end{pmatrix} = (a-\lambda)(d-\lambda) - bc = \lambda^2 - (a+d)\lambda + (ad - bc) = 0$$

This is $\lambda^2 - \text{tr}(A)\,\lambda + \det(A) = 0$, with roots:

$$\lambda = \frac{\text{tr}(A) \pm \sqrt{\text{tr}(A)^2 - 4\det(A)}}{2}$$

For larger matrices, there is no closed-form solution (Abel's impossibility theorem for degree $\geq 5$), which is why we need iterative algorithms like QR.

### Exercise 2 -- Characteristic Polynomial for 2x2 (Basic)

Compute the eigenvalues of a 2x2 symmetric matrix using the quadratic formula above, and verify they match `np.linalg.eig`.

In [5]:
# Exercise 2: Characteristic polynomial for 2x2 matrix

A2 = np.array([[5.0, 2.0],
               [2.0, 1.0]])

a, b, c, d = A2[0, 0], A2[0, 1], A2[1, 0], A2[1, 1]

# Compute trace and determinant
trace = a + d           # tr(A)
det = a * d - b * c     # det(A)

# Compute discriminant and roots using quadratic formula
discriminant = trace**2 - 4 * det
sqrt_disc = np.sqrt(discriminant + 0j)   # +0j handles potential complex case
lambda_1 = ((trace + sqrt_disc) / 2).real   # larger root
lambda_2 = ((trace - sqrt_disc) / 2).real   # smaller root

# Ensure lambda_1 >= lambda_2
if lambda_1 < lambda_2:
    lambda_1, lambda_2 = lambda_2, lambda_1

# Compare with numpy
lambdas_np, _ = np.linalg.eig(A2)

print('Matrix A2:')
print(A2)
print(f'\nTrace = {trace:.4f},  Det = {det:.4f}')
print(f'Discriminant = {discriminant:.4f}')
print(f'\nCharacteristic polynomial roots: lambda_1 = {lambda_1:.6f},  lambda_2 = {lambda_2:.6f}')
print(f'NumPy eigenvalues:               {np.sort(lambdas_np)[::-1]}')

Matrix A2:
[[5. 2.]
 [2. 1.]]

Trace = 6.0000,  Det = 1.0000
Discriminant = 32.0000

Characteristic polynomial roots: lambda_1 = 5.828427,  lambda_2 = 0.171573
NumPy eigenvalues:               [5.82842712 0.17157288]


In [6]:
# Verification cell -- do not modify
assert trace is not None, 'Compute trace'
assert det is not None, 'Compute det'
assert discriminant is not None, 'Compute discriminant'
assert lambda_1 is not None and lambda_2 is not None, 'Compute roots'
assert np.isclose(trace, a + d, atol=1e-8), 'Trace should be a+d'
assert np.isclose(det, a*d - b*c, atol=1e-8), 'Det should be ad-bc'
assert lambda_1 >= lambda_2, 'lambda_1 should be the larger root'
roots_manual = np.sort([lambda_1, lambda_2])[::-1]
roots_np = np.sort(lambdas_np.real)[::-1]
assert np.allclose(roots_manual, roots_np, atol=1e-6), \
    f'Roots do not match NumPy: manual={roots_manual}, np={roots_np}'
print('Exercise 2 PASSED: Characteristic polynomial roots match np.linalg.eig.')

Exercise 2 PASSED: Characteristic polynomial roots match np.linalg.eig.


---

## Part 3: The Spectral Theorem -- Why Symmetric Matrices Are Special

Goodfellow emphasizes a crucial fact: **every real symmetric matrix** has a particularly clean eigendecomposition.

**Spectral Theorem:** If $A$ is a real symmetric matrix ($A = A^\top$), then:
1. All eigenvalues are **real** (no complex values)
2. The eigenvectors can be chosen to be **mutually orthogonal**
3. $A$ can *always* be eigendecomposed as $A = Q\Lambda Q^\top$

where $Q$ is an **orthogonal matrix** (columns are orthonormal eigenvectors, so $Q^\top Q = I$) and $\Lambda = \text{diag}(\lambda_1, \ldots, \lambda_n)$.

**Why is $Q^\top = Q^{-1}$ so nice?**
- Orthogonal matrices preserve lengths and angles (they are pure rotations/reflections)
- $Q^{-1}$ is trivially $Q^\top$ -- no matrix inversion needed (huge computational savings)
- The decomposition $A = Q\Lambda Q^\top$ means: *rotate to eigenvector coordinates, scale along each axis, rotate back*

**Why do we care in ML?** Covariance matrices are always symmetric. Hessians of scalar functions are always symmetric. The graph Laplacian is always symmetric. Essentially, most matrices we eigendecompose in ML are symmetric, so we almost always get the clean version.

### Exercise 3 -- Spectral Theorem Verification

Verify the spectral theorem: given a symmetric matrix, show that `np.linalg.eigh` returns real eigenvalues and orthogonal eigenvectors, and that $A = Q\Lambda Q^\top$.

In [7]:
# Exercise 3: Spectral theorem for symmetric matrices

S = np.array([[4.0, 2.0, 1.0],
              [2.0, 5.0, 3.0],
              [1.0, 3.0, 6.0]])

# Verify symmetry
is_symmetric = np.allclose(S, S.T)

# Use eigh (specialized for symmetric/Hermitian matrices)
# eigh guarantees real eigenvalues in ascending order and orthonormal eigenvectors
eigvals_s, Q = np.linalg.eigh(S)

# Verify orthogonality: Q^T Q should be identity
QtQ = Q.T @ Q
is_orthogonal = np.allclose(QtQ, np.eye(3))

# Reconstruct: S = Q Lambda Q^T (NOT Q Lambda Q^{-1}, because Q^T = Q^{-1})
S_rec = Q @ np.diag(eigvals_s) @ Q.T

print('Symmetric matrix S:')
print(S)
print(f'\nS == S^T? {is_symmetric}')
print(f'\nEigenvalues (all real): {eigvals_s}')
print(f'\nQ^T Q (should be identity):')
print(QtQ)
print(f'\nQ^T Q is identity? {is_orthogonal}')
print(f'\nReconstruction S = Q Lambda Q^T:')
print(np.round(S_rec, 10))
print(f'\nReconstruction error: {np.max(np.abs(S - S_rec))}')

Symmetric matrix S:
[[4. 2. 1.]
 [2. 5. 3.]
 [1. 3. 6.]]

S == S^T? True

Eigenvalues (all real): [1.58481059 4.28498872 9.13020068]

Q^T Q (should be identity):
[[ 1.00000000e+00 -5.55111512e-17  2.77555756e-17]
 [-5.55111512e-17  1.00000000e+00  2.22044605e-16]
 [ 2.77555756e-17  2.22044605e-16  1.00000000e+00]]

Q^T Q is identity? True

Reconstruction S = Q Lambda Q^T:
[[4. 2. 1.]
 [2. 5. 3.]
 [1. 3. 6.]]

Reconstruction error: 3.5527136788005009e-15


In [8]:
# Verification cell -- do not modify
assert is_symmetric, 'S should be symmetric'
assert np.all(np.isreal(eigvals_s)), 'Eigenvalues of symmetric matrix must be real'
assert is_orthogonal, 'Eigenvectors of symmetric matrix must be orthogonal'
assert np.allclose(S, S_rec, atol=1e-10), 'S = Q Lambda Q^T reconstruction failed'
# Verify each eigenvector equation
for i in range(3):
    assert np.allclose(S @ Q[:, i], eigvals_s[i] * Q[:, i], atol=1e-10), \
        f'Eigenvector equation failed for i={i}'
print('Exercise 3 PASSED: Spectral theorem verified for symmetric matrix.')

Exercise 3 PASSED: Spectral theorem verified for symmetric matrix.


---

## Part 4: What Eigenvalues Tell You -- Matrix Classification

Goodfellow makes a critical point: the **signs of the eigenvalues** determine the nature of the matrix, which in turn determines the nature of critical points in optimization.

### Matrix Classification by Eigenvalue Signs

| Eigenvalue condition | Matrix type | Optimization meaning |
|---|---|---|
| All $\lambda_i > 0$ | **Positive definite (PD)** | Local minimum exists |
| All $\lambda_i \geq 0$ | **Positive semidefinite (PSD)** | Minimum in some directions, flat in others |
| All $\lambda_i < 0$ | **Negative definite (ND)** | Local maximum exists |
| Mixed signs | **Indefinite** | **Saddle point** |

### Two Fundamental Identities

These connect eigenvalues to familiar matrix quantities:

$$\text{tr}(A) = \sum_{i=1}^n \lambda_i \qquad \text{(trace = sum of eigenvalues)}$$

$$\det(A) = \prod_{i=1}^n \lambda_i \qquad \text{(determinant = product of eigenvalues)}$$

**Intuition for the determinant identity:** If the eigenvalues are the scale factors along each principal axis, then the determinant (which measures volume change) is the product of all scale factors. If any eigenvalue is zero, the matrix collapses a dimension and the determinant is zero.

### Exercise 4 -- Classify Matrices by Eigenvalue Signs

Construct three 3x3 symmetric matrices: one positive definite, one positive semidefinite (but not PD), and one indefinite. For each, verify:
- The classification via eigenvalue signs
- $\text{tr}(A) = \sum \lambda_i$
- $\det(A) = \prod \lambda_i$

In [9]:
# Exercise 4: Matrix classification via eigenvalue signs

# 1a. Construct a 3x3 Positive Definite matrix
#     Strategy: B @ B^T is always PSD; add epsilon*I to make it strictly PD
B_pd = np.array([[2.0, 1.0], [0.0, 1.0], [1.0, 1.0]])  # 3x2
A_pd = B_pd @ B_pd.T + np.eye(3)  # guarantees PD because B@B^T is PSD, +I shifts all eigenvalues up by 1

# 1b. Construct a 3x3 Positive Semidefinite matrix (with at least one zero eigenvalue)
#     Strategy: B @ B^T where B is 3x2 => rank at most 2, so at least one eigenvalue is 0
B_psd = np.array([[1.0, 1.0], [1.0, 0.0], [1.0, 1.0]])  # 3x2
A_psd = B_psd @ B_psd.T               # rank-2 => at least one eigenvalue is exactly 0

# 1c. Construct a 3x3 Indefinite matrix (mixed eigenvalue signs)
#     Diagonal with mixed signs guarantees mixed eigenvalues
A_indef = np.diag([3.0, -1.0, 2.0])

# 2. Compute eigenvalues for each (use eigh for symmetric matrices)
eigs_pd    = np.linalg.eigh(A_pd)[0]
eigs_psd   = np.linalg.eigh(A_psd)[0]
eigs_indef = np.linalg.eigh(A_indef)[0]

# 3. Verify trace and det relationships
trace_ok_pd    = np.isclose(np.trace(A_pd),    np.sum(eigs_pd),    atol=1e-6)
det_ok_pd      = np.isclose(np.linalg.det(A_pd), np.prod(eigs_pd), atol=1e-6)
trace_ok_psd   = np.isclose(np.trace(A_psd),   np.sum(eigs_psd),   atol=1e-6)
det_ok_psd     = np.isclose(np.linalg.det(A_psd), np.prod(eigs_psd), atol=1e-6)
trace_ok_indef = np.isclose(np.trace(A_indef), np.sum(eigs_indef), atol=1e-6)
det_ok_indef   = np.isclose(np.linalg.det(A_indef), np.prod(eigs_indef), atol=1e-6)

for name, A_mat, eigs in [('PD', A_pd, eigs_pd),
                       ('PSD', A_psd, eigs_psd),
                       ('Indefinite', A_indef, eigs_indef)]:
    print(f'\n--- {name} Matrix ---')
    print(f'  Eigenvalues: {np.round(eigs, 6)}')
    print(f'  All positive?        {np.all(eigs > 1e-10)} (PD if True)')
    print(f'  All non-negative?    {np.all(eigs >= -1e-10)} (PSD if True)')
    print(f'  Mixed signs?         {np.any(eigs > 1e-10) and np.any(eigs < -1e-10)} (Indefinite if True)')
    print(f'  Trace check:  trace={np.trace(A_mat):.6f}, sum(eigs)={np.sum(eigs):.6f}')
    print(f'  Det check:    det={np.linalg.det(A_mat):.6f}, prod(eigs)={np.prod(eigs):.6f}')


--- PD Matrix ---
  Eigenvalues: [0.585786 3.       5.414214]
  All positive?        True (PD if True)
  All non-negative?    True (PSD if True)
  Mixed signs?         False (Indefinite if True)
  Trace check:  trace=9.000000, sum(eigs)=9.000000
  Det check:    det=9.000000, prod(eigs)=9.000000

--- PSD Matrix ---
  Eigenvalues: [-0.       0.       5.333333]
  All positive?        False (PD if True)
  All non-negative?    True (PSD if True)
  Mixed signs?         False (Indefinite if True)
  Trace check:  trace=5.333333, sum(eigs)=5.333333
  Det check:    det=0.000000, prod(eigs)=-0.000000

--- Indefinite Matrix ---
  Eigenvalues: [-1.  2.  3.]
  All positive?        False (PD if True)
  All non-negative?    False (PSD if True)
  Mixed signs?         True (Indefinite if True)
  Trace check:  trace=4.000000, sum(eigs)=4.000000
  Det check:    det=-6.000000, prod(eigs)=-6.000000


In [10]:
# Verification cell -- do not modify
assert np.allclose(A_pd, A_pd.T), 'A_pd must be symmetric'
assert np.allclose(A_psd, A_psd.T), 'A_psd must be symmetric'
assert np.allclose(A_indef, A_indef.T), 'A_indef must be symmetric'
assert np.all(eigs_pd > 1e-10), 'PD matrix must have all positive eigenvalues'
assert np.all(eigs_psd >= -1e-10), 'PSD matrix must have all non-negative eigenvalues'
assert np.any(eigs_psd < 1e-10), 'PSD matrix should have at least one ~zero eigenvalue'
assert np.any(eigs_indef > 1e-10) and np.any(eigs_indef < -1e-10), 'Indefinite must have mixed signs'
assert trace_ok_pd and trace_ok_psd and trace_ok_indef, 'Trace identity failed'
assert det_ok_pd and det_ok_indef, 'Det identity failed for PD or Indefinite'
print('Exercise 4 PASSED: Matrix classification and trace/det identities verified.')

Exercise 4 PASSED: Matrix classification and trace/det identities verified.


---

## Part 5: Quadratic Forms and Their Geometry

### The Quadratic Form

A **quadratic form** is a scalar-valued function:

$$f(\mathbf{x}) = \mathbf{x}^\top A \mathbf{x}$$

where $A$ is a symmetric matrix. This is the second-order Taylor term that dominates near critical points. In optimization, the Hessian $H$ plays the role of $A$, and the quadratic form $\mathbf{x}^\top H \mathbf{x}$ tells you the curvature of the loss surface.

### The Connection to Eigendecomposition

Substitute $A = Q\Lambda Q^\top$ into the quadratic form:

$$f(\mathbf{x}) = \mathbf{x}^\top Q \Lambda Q^\top \mathbf{x} = (Q^\top \mathbf{x})^\top \Lambda (Q^\top \mathbf{x}) = \mathbf{y}^\top \Lambda \mathbf{y} = \sum_{i=1}^n \lambda_i y_i^2$$

where $\mathbf{y} = Q^\top \mathbf{x}$ are coordinates in the eigenvector basis.

**This is the key insight:** In the eigenvector coordinate system, the quadratic form completely decouples. Each eigenvalue $\lambda_i$ controls the curvature along its corresponding eigenvector direction.

### What the Level Sets Look Like

The level set $\mathbf{x}^\top A \mathbf{x} = c$ in eigenvector coordinates becomes $\sum \lambda_i y_i^2 = c$:

- **PD ($\lambda_i > 0$):** Ellipsoid centered at origin -- bowl shape. The level sets are ellipses whose axes are the eigenvectors, with semi-axis lengths proportional to $1/\sqrt{\lambda_i}$.
- **Indefinite (mixed signs):** Hyperboloid -- saddle shape. Some directions curve up, others curve down.
- **PSD ($\lambda_i \geq 0$, some zero):** The ellipsoid extends to infinity along the zero-eigenvalue directions (flat valley).

### Exercise 5 -- Quadratic Form Visualization

Visualize the quadratic forms $f(\mathbf{x}) = \mathbf{x}^\top A \mathbf{x}$ for a PD matrix and an indefinite matrix. Show that the eigenvectors are the principal axes.

In [11]:
# Exercise 5: Quadratic form visualization

A_pd5  = np.array([[3.0, 1.0],
                   [1.0, 2.0]])   # Positive Definite

A_indef5 = np.array([[ 2.0, 1.0],
                     [ 1.0, -2.0]])   # Indefinite

# Create grid for contour plotting
x_range = np.linspace(-2.0, 2.0, 400)
y_range = np.linspace(-2.0, 2.0, 400)
X, Y = np.meshgrid(x_range, y_range)

# 1. Compute quadratic form Z = x^T A x on the grid for both matrices
#    Vectorized: Z = A[0,0]*X^2 + (A[0,1]+A[1,0])*X*Y + A[1,1]*Y^2
Z_pd    = A_pd5[0,0]*X**2    + (A_pd5[0,1]    + A_pd5[1,0])    * X * Y + A_pd5[1,1]*Y**2
Z_indef = A_indef5[0,0]*X**2 + (A_indef5[0,1] + A_indef5[1,0]) * X * Y + A_indef5[1,1]*Y**2

# 2. Get eigenvalues and eigenvectors for both matrices
eigs_pd5,    vecs_pd5    = np.linalg.eigh(A_pd5)
eigs_indef5, vecs_indef5 = np.linalg.eigh(A_indef5)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, Z, A_mat, eigs, vecs, title in [
    (axes[0], Z_pd,    A_pd5,    eigs_pd5,    vecs_pd5,    'PD Matrix: Bowl ($x^T A x = c$)'),
    (axes[1], Z_indef, A_indef5, eigs_indef5, vecs_indef5, 'Indefinite: Saddle ($x^T A x = c$)')
]:
    ax.set_xlim(-2, 2); ax.set_ylim(-2, 2)
    ax.set_aspect('equal')
    ax.axhline(0, color='k', lw=0.8); ax.axvline(0, color='k', lw=0.8)
    ax.set_xlabel('$x_1$', fontsize=12); ax.set_ylabel('$x_2$', fontsize=12)
    ax.set_title(title, fontsize=11)

    ax.contour(X, Y, Z, levels=[1.0], colors='#2980b9', linewidths=2)
    ax.contour(X, Y, Z, levels=[-1.0], colors='#e74c3c', linewidths=2, linestyles='dashed')
    ax.contourf(X, Y, Z, levels=20, cmap='RdBu_r', alpha=0.3)

    # Draw eigenvectors scaled for visibility
    colors_ev = ['#e67e22', '#27ae60']
    for i in range(2):
        v = vecs[:, i]
        scale = 1.5
        ax.annotate('', xy=scale*v, xytext=(0, 0),
                    arrowprops=dict(arrowstyle='->', color=colors_ev[i], lw=2.5))
        ax.text(scale*v[0]*1.15, scale*v[1]*1.15,
                f'$\\lambda={eigs[i]:.2f}$', fontsize=10, color=colors_ev[i], fontweight='bold')

plt.tight_layout()
plt.show()

<Figure size 1400x600 with 2 Axes>

**Reading the plots:**

- **Left (PD):** Both eigenvalues are positive, so $\mathbf{x}^\top A \mathbf{x} > 0$ for all nonzero $\mathbf{x}$. The level set $\mathbf{x}^\top A \mathbf{x} = 1$ is an ellipse. The eigenvectors point along the axes of the ellipse. The larger eigenvalue corresponds to the shorter semi-axis (higher curvature = tighter ellipse). This is a **bowl** -- gradient descent will find the minimum at the origin.

- **Right (Indefinite):** One eigenvalue is positive, one is negative. The level sets form hyperbolas. Along the positive-eigenvalue eigenvector, the quadratic increases; along the negative-eigenvalue eigenvector, it decreases. The origin is a **saddle point** -- not a minimum or maximum.

**This is exactly what happens with the Hessian at a critical point of a loss function.** PD Hessian means you found a local minimum. Indefinite Hessian means saddle point (very common in deep learning -- most critical points in high dimensions are saddle points, not minima).

In [12]:
# Verification cell -- do not modify
# Spot-check the quadratic form computation
test_x = np.array([1.0, 0.5])
expected_pd = test_x @ A_pd5 @ test_x
grid_val_pd = A_pd5[0,0]*1.0**2 + (A_pd5[0,1]+A_pd5[1,0])*1.0*0.5 + A_pd5[1,1]*0.5**2
assert np.isclose(expected_pd, grid_val_pd), 'Quadratic form computation error for PD'
assert np.all(eigs_pd5 > 0), 'A_pd5 should be positive definite'
assert eigs_indef5[0] * eigs_indef5[-1] < 0, 'A_indef5 should be indefinite'
print('Exercise 5 PASSED: Quadratic form computation and eigendecomposition verified.')

Exercise 5 PASSED: Quadratic form computation and eigendecomposition verified.


---

## Part 6: Power Iteration -- The Simplest Eigenvalue Algorithm

### The Algorithm

**Power iteration** finds the dominant (largest-magnitude) eigenvalue and its eigenvector. It is beautifully simple:

1. Start with a random unit vector $\mathbf{b}_0$
2. Repeat: $\mathbf{b}_{k+1} = \dfrac{A \mathbf{b}_k}{\|A \mathbf{b}_k\|}$
3. Estimate eigenvalue via the **Rayleigh quotient**: $\lambda_k = \mathbf{b}_k^\top A \mathbf{b}_k$

### Why It Works

Expand $\mathbf{b}_0$ in the eigenvector basis: $\mathbf{b}_0 = c_1 \mathbf{v}_1 + c_2 \mathbf{v}_2 + \cdots$

After $k$ multiplications by $A$:

$$A^k \mathbf{b}_0 = c_1 \lambda_1^k \mathbf{v}_1 + c_2 \lambda_2^k \mathbf{v}_2 + \cdots$$

If $|\lambda_1| > |\lambda_2| \geq \cdots$, then $\lambda_1^k$ grows faster than all other terms. Factor it out:

$$A^k \mathbf{b}_0 = \lambda_1^k \left[ c_1 \mathbf{v}_1 + c_2 \left(\frac{\lambda_2}{\lambda_1}\right)^k \mathbf{v}_2 + \cdots \right]$$

Since $|\lambda_2/\lambda_1| < 1$, the ratio $\left(\frac{\lambda_2}{\lambda_1}\right)^k \to 0$ as $k \to \infty$. So $A^k \mathbf{b}_0$ converges to a multiple of $\mathbf{v}_1$.

**Convergence rate:** Linear, proportional to $|\lambda_2 / \lambda_1|$. Closer eigenvalues = slower convergence.

### ML Connection: PageRank

Google's original PageRank algorithm is literally power iteration on the web's link matrix. The dominant eigenvector of the transition matrix gives the steady-state probability distribution -- the "importance" of each page.

### Exercise 6 -- Implement Power Iteration (Intermediate)

Implement power iteration from scratch. Return the dominant eigenvalue, eigenvector, convergence history, and number of iterations.

In [13]:
# Exercise 6: Power iteration from scratch

def power_iteration(A, max_iter=200, tol=1e-10):
    """Find the dominant eigenvalue and eigenvector via power iteration.
    
    Args:
        A: Square matrix (n x n)
        max_iter: Maximum number of iterations
        tol: Convergence tolerance for eigenvalue change
    
    Returns:
        eigenvalue: The dominant (largest magnitude) eigenvalue
        eigenvector: The corresponding unit eigenvector
        history: List of eigenvalue estimates at each iteration
        n_iter: Number of iterations until convergence
    """
    n = A.shape[0]
    
    # Step 1: Start with random unit vector
    b = np.random.randn(n)
    b = b / np.linalg.norm(b)
    
    history = []
    eigenvalue = 0.0
    n_iter = 0
    
    for k in range(max_iter):
        # Step 2: Multiply by A
        Ab = A @ b
        
        # Step 3: Normalize
        b_new = Ab / np.linalg.norm(Ab)
        
        # Step 4: Rayleigh quotient for eigenvalue estimate
        eigenvalue_new = b_new @ A @ b_new
        history.append(eigenvalue_new)
        
        # Step 5: Check convergence
        if k > 0 and abs(eigenvalue_new - eigenvalue) < tol:
            eigenvalue = eigenvalue_new
            b = b_new
            n_iter = k + 1
            break
        
        eigenvalue = eigenvalue_new
        b = b_new
        n_iter = k + 1
    
    return eigenvalue, b, history, n_iter


# Test on a symmetric matrix
A3 = np.array([[4.0, 1.0, 0.5],
               [1.0, 3.0, 0.8],
               [0.5, 0.8, 2.0]])

pi_eigenvalue, pi_eigenvector, pi_history, pi_niter = power_iteration(A3)

# Compare with numpy
true_eigvals, true_eigvecs = np.linalg.eigh(A3)
true_dominant = true_eigvals[-1]  # eigh returns in ascending order

print(f'Matrix A3:')
print(A3)
print(f'\nPower Iteration dominant eigenvalue: {pi_eigenvalue:.8f}')
print(f'True dominant eigenvalue (numpy):    {true_dominant:.8f}')
print(f'Absolute error: {abs(pi_eigenvalue - true_dominant):.2e}')
print(f'Converged in {pi_niter} iterations')

# Plot convergence
fig, ax = plt.subplots(1, 1, figsize=(10, 4))
errors = [abs(h - true_dominant) for h in pi_history]
ax.semilogy(range(1, len(errors)+1), errors, 'o-', color='#2980b9', markersize=4)
ax.axhline(y=1e-10, color='r', linestyle='--', alpha=0.5, label='Tolerance')
ax.set_xlabel('Iteration', fontsize=12)
ax.set_ylabel('|estimate - true eigenvalue|', fontsize=12)
ax.set_title('Power Iteration Convergence', fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

Matrix A3:
[[4.  1.  0.5]
 [1.  3.  0.8]
 [0.5 0.8 2. ]]

Power Iteration dominant eigenvalue: 4.87329392
True dominant eigenvalue (numpy):    4.87329392
Absolute error: 1.68e-11
Converged in 18 iterations


<Figure size 1000x400 with 1 Axes>

In [14]:
# Verification cell -- do not modify
assert callable(power_iteration), 'power_iteration must be a function'
assert np.isclose(pi_eigenvalue, true_dominant, atol=1e-6), \
    f'Eigenvalue mismatch: got {pi_eigenvalue:.8f}, expected {true_dominant:.8f}'
# Check eigenvector (up to sign)
true_dominant_vec = true_eigvecs[:, -1]
assert np.allclose(np.abs(pi_eigenvector @ true_dominant_vec), 1.0, atol=1e-4), \
    'Eigenvector direction mismatch'
assert len(pi_history) > 0, 'History should not be empty'
assert pi_niter <= 200, 'Should converge within max_iter'
print('Exercise 6 PASSED: Power iteration converges to the dominant eigenvalue.')

Exercise 6 PASSED: Power iteration converges to the dominant eigenvalue.


---

## Part 7: The QR Algorithm -- Computing All Eigenvalues

### Motivation

Power iteration only finds the **dominant** eigenvalue. What if we need all of them? The **QR algorithm** is the workhorse of numerical linear algebra for this purpose. It is what `np.linalg.eig` uses under the hood (via LAPACK).

### The Basic QR Algorithm

1. Start with $A_0 = A$
2. At each step $k$:
   - Compute the QR decomposition: $A_k = Q_k R_k$
   - Form the next iterate: $A_{k+1} = R_k Q_k$
3. As $k \to \infty$, $A_k$ converges to an upper triangular (or quasi-triangular) matrix whose diagonal entries are the eigenvalues.

### Why Does This Work?

The key observation: $A_{k+1} = R_k Q_k = Q_k^\top (Q_k R_k) Q_k = Q_k^\top A_k Q_k$.

So each iteration is a **similarity transformation** -- it preserves eigenvalues while driving the matrix toward triangular form. The off-diagonal elements shrink at a rate related to the ratios of eigenvalues (just like power iteration).

**In practice**, the QR algorithm is accelerated with:
- **Shifts** (Wilkinson shift): dramatically faster convergence from linear to cubic
- **Hessenberg reduction**: reduce to almost-triangular form first (saves computation)
- **Deflation**: once an eigenvalue converges, peel it off and work on a smaller matrix

We implement the basic (unshifted) version to build intuition.

### Exercise 7 -- Implement the Basic QR Algorithm (Intermediate)

Implement the basic QR iteration. Watch the off-diagonal elements shrink to zero as the diagonal converges to the eigenvalues.

In [15]:
# Exercise 7: Basic QR algorithm

def qr_algorithm(A, max_iter=200, tol=1e-10):
    """Compute all eigenvalues using the basic QR algorithm.
    
    Args:
        A: Square symmetric matrix (n x n)
        max_iter: Maximum number of QR iterations
        tol: Convergence tolerance (max off-diagonal element)
    
    Returns:
        eigenvalues: Array of eigenvalues (diagonal of converged matrix)
        A_k: The final (nearly diagonal) matrix
        off_diag_history: List of max off-diagonal magnitude at each iteration
        n_iter: Number of iterations
    """
    n = A.shape[0]
    A_k = A.copy().astype(float)
    off_diag_history = []
    n_iter = 0
    
    for k in range(max_iter):
        # Step 1: QR decomposition
        Q_k, R_k = np.linalg.qr(A_k)
        
        # Step 2: Form next iterate A_{k+1} = R_k @ Q_k
        A_k = R_k @ Q_k
        
        # Track off-diagonal magnitude
        off_diag = np.max(np.abs(A_k - np.diag(np.diag(A_k))))
        off_diag_history.append(off_diag)
        n_iter = k + 1
        
        if off_diag < tol:
            break
    
    eigenvalues = np.diag(A_k)
    return eigenvalues, A_k, off_diag_history, n_iter


# Test on same symmetric matrix
A4 = np.array([[4.0, 1.0, 0.5],
               [1.0, 3.0, 0.8],
               [0.5, 0.8, 2.0]])

qr_eigenvalues, qr_A_final, qr_history, qr_niter = qr_algorithm(A4)

# Compare with numpy
true_eigs = np.sort(np.linalg.eigh(A4)[0])

print(f'Matrix A4:')
print(A4)
print(f'\nQR Algorithm converged in {qr_niter} iterations')
print(f'\nFinal A_k (should be nearly diagonal for symmetric input):')
print(qr_A_final)
print(f'\nDiagonal of final A_k (eigenvalues): {qr_eigenvalues}')
print(f'True eigenvalues (numpy):            {true_eigs}')
print(f'\nMax off-diagonal magnitude: {qr_history[-1]:.3e}')

# Plot convergence
fig, ax = plt.subplots(1, 1, figsize=(10, 4))
ax.semilogy(range(1, len(qr_history)+1), qr_history, 'o-', color='#e74c3c', markersize=4)
ax.axhline(y=1e-10, color='gray', linestyle='--', alpha=0.5, label='Tolerance')
ax.set_xlabel('QR Iteration', fontsize=12)
ax.set_ylabel('Max off-diagonal element', fontsize=12)
ax.set_title('QR Algorithm: Off-Diagonal Elements Converge to Zero', fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

Matrix A4:
[[4.  1.  0.5]
 [1.  3.  0.8]
 [0.5 0.8 2. ]]

QR Algorithm converged in 27 iterations

Final A_k (should be nearly diagonal for symmetric input):
[[ 4.87329392e+00  4.17459498e-12 -2.52162377e-17]
 [ 4.17459498e-12  2.82757988e+00  2.46037727e-06]
 [-2.52162377e-17  2.46037727e-06  1.29912620e+00]]

Diagonal of final A_k (eigenvalues): [4.87329392 2.82757988 1.2991262 ]
True eigenvalues (numpy):            [1.2991262  2.82757988 4.87329392]

Max off-diagonal magnitude: 2.460e-06


<Figure size 1000x400 with 1 Axes>

In [16]:
# Verification cell -- do not modify
assert callable(qr_algorithm), 'qr_algorithm must be a function'
# Compare sorted eigenvalues
qr_sorted = np.sort(qr_eigenvalues)
true_sorted = np.sort(true_eigs)
assert np.allclose(qr_sorted, true_sorted, atol=1e-4), \
    f'Eigenvalues mismatch: got {qr_sorted}, expected {true_sorted}'
assert len(qr_history) > 0, 'History should not be empty'
# Off-diagonal should decrease
assert qr_history[-1] < qr_history[0], 'Off-diagonal should decrease over iterations'
print('Exercise 7 PASSED: QR algorithm finds all eigenvalues.')

Exercise 7 PASSED: QR algorithm finds all eigenvalues.


### Power Iteration vs QR: A Comparison

| Algorithm | Purpose | Convergence | Cost per iteration |
|---|---|---|---|
| Power iteration | Dominant eigenvalue only | Linear, rate $|\lambda_2/\lambda_1|$ | $O(n^2)$ (matrix-vector multiply) |
| Inverse power iteration | Eigenvalue closest to a shift $\sigma$ | Linear, rate of shifted inverse | $O(n^3)$ (solve linear system) |
| QR algorithm (basic) | All eigenvalues simultaneously | Linear (quadratic/cubic with shifts) | $O(n^3)$ (QR factorization) |

**When to use which:**
- Power iteration: when you only need the top-$k$ eigenvalues of a *huge* sparse matrix (e.g., PageRank, PCA on big data)
- QR algorithm: when you need *all* eigenvalues of a moderate-sized dense matrix (e.g., full Hessian analysis)

---

## Part 8: ML Connections -- Why This All Matters

### Connection 1: PCA (Principal Component Analysis)

PCA finds the directions of maximum variance in data. It is literally the eigendecomposition of the covariance matrix $\Sigma = \frac{1}{n}X^\top X$.

- Eigenvectors of $\Sigma$ = principal components (directions of maximum variance)
- Eigenvalues of $\Sigma$ = variance along each principal component
- Projecting onto the top-$k$ eigenvectors gives the best rank-$k$ approximation

### Connection 2: Hessian and Loss Landscape Curvature

The Hessian $H$ of the loss function $\mathcal{L}(\theta)$ is a symmetric matrix. At a critical point ($\nabla \mathcal{L} = 0$):

- **All eigenvalues of $H > 0$**: local minimum (the loss curves up in every direction)
- **All eigenvalues of $H < 0$**: local maximum (the loss curves down in every direction)
- **Mixed signs**: saddle point (some directions curve up, others curve down)

In deep learning, the Hessian of the loss typically has many near-zero eigenvalues (flat directions) and a few large ones. This makes optimization tricky -- the loss landscape is like a long, narrow valley.

### Connection 3: Condition Number and Optimization Difficulty

The **condition number** of a PD matrix is:

$$\kappa(A) = \frac{\lambda_{\max}}{\lambda_{\min}}$$

This determines how hard it is to optimize a quadratic $\mathbf{x}^\top A \mathbf{x}$ with gradient descent:
- $\kappa \approx 1$: nearly circular contours, gradient descent converges fast
- $\kappa \gg 1$: elongated elliptical contours, gradient descent zigzags and converges slowly

The optimal step size for gradient descent on a quadratic is $\frac{2}{\lambda_{\max} + \lambda_{\min}}$, and the number of iterations scales as $O(\kappa \log(1/\epsilon))$.

### Connection 4: Spectral Clustering

Spectral clustering uses the eigenvectors of the **graph Laplacian** $L = D - W$ (where $W$ is the adjacency matrix and $D$ is the degree matrix). The bottom eigenvectors of $L$ reveal cluster structure -- points in the same cluster have similar eigenvector coordinates.

### Exercise 8 -- PCA via Eigendecomposition

Implement PCA on a 2D dataset by eigendecomposing the covariance matrix. Project the data onto the first principal component.

In [17]:
# Exercise 8: PCA via eigendecomposition of covariance matrix

# Generate 2D data with correlation
np.random.seed(42)
n_samples = 200
mean = np.array([0.0, 0.0])
cov_true = np.array([[4.0, 2.0],
                     [2.0, 1.5]])
X_data = np.random.multivariate_normal(mean, cov_true, n_samples)

# Step 1: Center the data (subtract mean)
X_centered = X_data - X_data.mean(axis=0)

# Step 2: Compute covariance matrix
cov_matrix = (X_centered.T @ X_centered) / (n_samples - 1)

# Step 3: Eigendecompose the covariance matrix
pca_eigvals, pca_eigvecs = np.linalg.eigh(cov_matrix)

# Step 4: Sort by descending eigenvalue (eigh returns ascending)
sort_idx = np.argsort(pca_eigvals)[::-1]
pca_eigvals = pca_eigvals[sort_idx]
pca_eigvecs = pca_eigvecs[:, sort_idx]

# Step 5: Project data onto first principal component
pc1 = pca_eigvecs[:, 0]  # first principal component direction
X_proj_1d = X_centered @ pc1  # project onto PC1 (scalar per point)
X_proj_2d = np.outer(X_proj_1d, pc1)  # reconstruct in 2D

# Variance explained
var_explained = pca_eigvals / np.sum(pca_eigvals) * 100

# Condition number
kappa = pca_eigvals[0] / pca_eigvals[-1]

print('Covariance matrix:')
print(cov_matrix)
print(f'\nEigenvalues (variances): {pca_eigvals}')
print(f'Eigenvectors (principal components):')
print(pca_eigvecs)
print(f'\nVariance explained by PC1: {var_explained[0]:.2f}%')
print(f'Variance explained by PC2: {var_explained[1]:.2f}%')
print(f'\nCondition number kappa = lambda_max / lambda_min = {kappa:.2f}')
print(f'  --> {"Well-conditioned" if kappa < 10 else "Moderate ill-conditioning" if kappa < 100 else "Severely ill-conditioned"}')

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: Data + principal components
ax = axes[0]
ax.scatter(X_centered[:, 0], X_centered[:, 1], alpha=0.3, s=15, color='gray')
colors_pc = ['#e74c3c', '#2980b9']
for i in range(2):
    v = pca_eigvecs[:, i] * np.sqrt(pca_eigvals[i]) * 2  # scale by std for visibility
    ax.annotate('', xy=v, xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color=colors_pc[i], lw=3))
    ax.text(v[0]*1.15, v[1]*1.15,
            f'PC{i+1} ($\\lambda={pca_eigvals[i]:.2f}$)',
            fontsize=11, color=colors_pc[i], fontweight='bold')
ax.set_aspect('equal')
ax.set_title('Data + Principal Components', fontsize=13)
ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')

# Right: Projection onto PC1
ax = axes[1]
ax.scatter(X_centered[:, 0], X_centered[:, 1], alpha=0.15, s=15, color='gray', label='Original')
ax.scatter(X_proj_2d[:, 0], X_proj_2d[:, 1], alpha=0.5, s=15, color='#e74c3c', label='Projected onto PC1')
# Draw projection lines for a few points
for i in range(0, n_samples, 20):
    ax.plot([X_centered[i, 0], X_proj_2d[i, 0]],
            [X_centered[i, 1], X_proj_2d[i, 1]],
            'k-', alpha=0.15, lw=0.5)
ax.set_aspect('equal')
ax.set_title(f'PCA Projection (PC1 captures {var_explained[0]:.1f}% variance)', fontsize=12)
ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

Covariance matrix:
[[3.65594068 1.67979422]
 [1.67979422 1.17965397]]

Eigenvalues (variances): [0.38775296 4.44784169]
Eigenvectors (principal components):
[[-0.42009934 -0.90751653]
 [ 0.90751653 -0.42009934]]

Variance explained by PC1: 91.98%
Variance explained by PC2: 8.02%

Condition number kappa = lambda_max / lambda_min = 11.47
  --> Moderate ill-conditioning


<Figure size 1200x500 with 2 Axes>

In [18]:
# Verification cell -- do not modify
assert cov_matrix.shape == (2, 2), 'Covariance matrix should be 2x2'
assert np.allclose(cov_matrix, cov_matrix.T), 'Covariance matrix should be symmetric'
assert pca_eigvals[0] >= pca_eigvals[1], 'Eigenvalues should be in descending order'
assert np.all(pca_eigvals > 0), 'Covariance eigenvalues should be positive'
assert np.isclose(np.sum(var_explained), 100.0), 'Variance explained should sum to 100%'
assert var_explained[0] > 50, 'PC1 should explain majority of variance'
assert kappa > 1, 'Condition number should be > 1'
# Verify projection is correct
assert X_proj_1d.shape == (n_samples,), 'Projected data should be 1D'
assert X_proj_2d.shape == (n_samples, 2), 'Reconstructed projection should be 2D'
# Each projected point should lie on the PC1 line
for i in range(n_samples):
    assert np.isclose(np.abs(np.dot(X_proj_2d[i], pc1)), np.linalg.norm(X_proj_2d[i]), atol=1e-10) or \
           np.linalg.norm(X_proj_2d[i]) < 1e-10, \
        f'Point {i} not on PC1 line'
print('Exercise 8 PASSED: PCA via eigendecomposition verified.')

Exercise 8 PASSED: PCA via eigendecomposition verified.


### Exercise 9 -- Condition Number and Gradient Descent Difficulty

Visualize how the condition number $\kappa = \lambda_{\max}/\lambda_{\min}$ affects gradient descent on a quadratic $f(\mathbf{x}) = \frac{1}{2}\mathbf{x}^\top A \mathbf{x}$. Compare a well-conditioned case ($\kappa \approx 1$) with a poorly-conditioned case ($\kappa \gg 1$).

In [19]:
# Exercise 9: Condition number and gradient descent

def gradient_descent_quadratic(A, x0, n_steps=500, tol=1e-8):
    """Gradient descent on f(x) = 0.5 * x^T A x.
    Gradient: A @ x. Optimal step size for quadratic: 2 / (lmax + lmin)."""
    eigs = np.linalg.eigvalsh(A)
    lr = 2.0 / (eigs[-1] + eigs[0])  # optimal step size
    
    x = x0.copy()
    trajectory = [x.copy()]
    
    for step in range(n_steps):
        grad = A @ x
        x = x - lr * grad
        trajectory.append(x.copy())
        if np.linalg.norm(grad) < tol:
            break
    
    return np.array(trajectory)

# Well-conditioned: kappa ~ 2
A_well = np.array([[2.0, 0.0],
                   [0.0, 1.0]])  # kappa = 2/1 = 2

# Poorly conditioned: kappa = 50
A_poor = np.array([[50.0, 0.0],
                   [0.0,  1.0]])  # kappa = 50/1 = 50

x0 = np.array([4.0, 4.0])
traj_well = gradient_descent_quadratic(A_well, x0)
traj_poor = gradient_descent_quadratic(A_poor, x0)

kappa_well = np.linalg.eigvalsh(A_well)[-1] / np.linalg.eigvalsh(A_well)[0]
kappa_poor = np.linalg.eigvalsh(A_poor)[-1] / np.linalg.eigvalsh(A_poor)[0]

print(f'Well-conditioned:   kappa = {kappa_well:.2f}, converged in {len(traj_well)-1} steps')
print(f'Poorly-conditioned: kappa = {kappa_poor:.2f}, converged in {len(traj_poor)-1} steps')

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, A_mat, traj, kap, title in [
    (axes[0], A_well, traj_well, kappa_well, f'Well-conditioned ($\\kappa = {kappa_well:.0f}$)'),
    (axes[1], A_poor, traj_poor, kappa_poor, f'Poorly-conditioned ($\\kappa = {kappa_poor:.0f}$)')
]:
    # Contours of f(x) = 0.5 x^T A x
    x_r = np.linspace(-5, 5, 200)
    y_r = np.linspace(-5, 5, 200)
    Xg, Yg = np.meshgrid(x_r, y_r)
    Zg = 0.5 * (A_mat[0,0]*Xg**2 + (A_mat[0,1]+A_mat[1,0])*Xg*Yg + A_mat[1,1]*Yg**2)
    ax.contour(Xg, Yg, Zg, levels=20, cmap='Blues', alpha=0.6)
    
    # Plot trajectory
    ax.plot(traj[:, 0], traj[:, 1], 'o-', color='#e74c3c', markersize=2, lw=1, alpha=0.7)
    ax.plot(traj[0, 0], traj[0, 1], 'go', markersize=8, label='Start')
    ax.plot(traj[-1, 0], traj[-1, 1], 'r*', markersize=12, label='End')
    ax.set_title(f'{title}\n{len(traj)-1} steps', fontsize=12)
    ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
    ax.set_aspect('equal')
    ax.legend(fontsize=10)
    ax.set_xlim(-5, 5); ax.set_ylim(-5, 5)

plt.tight_layout()
plt.show()

Well-conditioned:   kappa = 2.00, converged in 24 steps
Poorly-conditioned: kappa = 50.00, converged in 461 steps


<Figure size 1400x600 with 2 Axes>

In [20]:
# Verification cell -- do not modify
assert kappa_well < kappa_poor, 'Well-conditioned should have smaller kappa'
assert len(traj_well) < len(traj_poor), 'Well-conditioned should converge faster'
# Both should converge near the origin
assert np.linalg.norm(traj_well[-1]) < 0.1, 'Well-conditioned should converge to origin'
assert np.linalg.norm(traj_poor[-1]) < 0.1, 'Poorly-conditioned should converge to origin'
print('Exercise 9 PASSED: Condition number governs GD convergence speed.')

Exercise 9 PASSED: Condition number governs GD convergence speed.


---

## Summary: Section 2.7 Key Takeaways

### Definitions

- **Eigenvector/eigenvalue:** $A\mathbf{v} = \lambda \mathbf{v}$. The matrix only *scales* the eigenvector; $\lambda$ is the scale factor.
- **Eigendecomposition:** $A = V\,\text{diag}(\boldsymbol{\lambda})\,V^{-1}$ (requires $n$ independent eigenvectors).
- **Spectral Theorem:** Real symmetric $\Rightarrow$ $A = Q\Lambda Q^\top$ with real $\lambda_i$ and orthogonal $Q$.

### What Eigenvalues Reveal

| Property | Formula |
|---|---|
| Trace | $\text{tr}(A) = \sum \lambda_i$ |
| Determinant | $\det(A) = \prod \lambda_i$ |
| Positive definite | All $\lambda_i > 0$ (bowl-shaped, minimum exists) |
| Positive semidefinite | All $\lambda_i \geq 0$ |
| Indefinite | Mixed signs (saddle point) |
| Condition number | $\kappa = \lambda_{\max}/\lambda_{\min}$ |

### Quadratic Forms

- $f(\mathbf{x}) = \mathbf{x}^\top A \mathbf{x} = \sum \lambda_i y_i^2$ in eigenvector coordinates
- Eigenvectors = principal axes of the quadratic
- Eigenvalues = curvature along each axis
- PD $\Rightarrow$ bowl (minimum), Indefinite $\Rightarrow$ saddle

### Algorithms

| Algorithm | Purpose | Convergence |
|---|---|---|
| Power iteration | Dominant eigenvalue | Linear, rate $|\lambda_2/\lambda_1|$ |
| Inverse power iteration | Eigenvalue closest to $\sigma$ | Linear, rate of shifted inverse |
| QR algorithm | All eigenvalues simultaneously | Quadratic (with shifts) |

### ML Connections

| Application | Role of Eigendecomposition |
|---|---|
| **PCA** | Eigenvectors of covariance = principal components; eigenvalues = variances |
| **Hessian analysis** | Eigenvalue signs determine if critical point is min/max/saddle |
| **Condition number** | $\kappa = \lambda_{\max}/\lambda_{\min}$ determines GD convergence speed |
| **Spectral clustering** | Bottom eigenvectors of graph Laplacian reveal cluster structure |
| **PageRank** | Dominant eigenvector of web transition matrix = page importance |